In [ ]:
# Cell 1: Imports
import re
import pdfplumber
import pandas as pd
from pathlib import Path

In [ ]:
# Cell 2: SET YOUR PATH HERE
# ========================================
# Change this path to your invoice folder
# ========================================
INVOICE_DIR = Path("Aug 25")

# List all PDF files (use set to avoid duplicates from case-insensitive matching)
pdf_files = list(set(INVOICE_DIR.glob("*.pdf")) | set(INVOICE_DIR.glob("*.PDF")))
print(f"Found {len(pdf_files)} PDF files in: {INVOICE_DIR}")

Found 1470 PDF files in: Amazon Data Extraction\Amz June\bulkInvoice_2025_06_29 12_00_00 AM IST - 2025_06_30 11_59_59 PM IST


In [70]:
# Cell 3: Helper Functions

def clean_amount(x):
    """Convert string amount to float (removes commas)"""
    return float(x.replace(",", ""))

def extract_text(pdf_path):
    """Extract full text from a PDF file."""
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text

In [71]:
# Cell 4: SKU Extraction Function

def extract_sku(full_text):
    """
    Extract SKU from Amazon invoice.
    Handles multiple formats:
    - B0xxxxxxxx ( HM0627 )
    - LG0995 | B0BNV9V36C
    - B0BNV9V36C ( 18980-28 )
    - SKU on separate line after ASIN
    - Gardena model numbers like 18315-20
    """
    # Pattern 1: ASIN followed by ( SKU ) where SKU starts with letters
    match = re.search(r"B[A-Z0-9]{9,10}\s*\(\s*([A-Z]{2}[A-Z0-9\-]+)\s*", full_text)
    if match:
        sku = match.group(1).strip()
        sku = re.sub(r"\s+\d+$", "", sku)
        return sku
    
    # Pattern 2: SKU BEFORE ASIN (like "LG0995 | B0BNV9V36C")
    match = re.search(r"((?:HM|LG|P)\d+(?:-[A-Z0-9]+)?)\s*\|?\s*B[A-Z0-9]{9,10}", full_text)
    if match:
        return match.group(1).strip()
    
    # Pattern 3: ASIN followed by ( alphanumeric-dash code )
    match = re.search(r"B[A-Z0-9]{9,10}\s*\(\s*([A-Z0-9]+-[A-Z0-9]+)\s*\)", full_text)
    if match:
        sku = match.group(1).strip()
        if not sku.startswith("₹"):
            return sku
    
    # Pattern 4: SKU on separate line after ASIN line
    lines = full_text.splitlines()
    for i, line in enumerate(lines):
        if re.search(r"B[A-Z0-9]{9,10}\s*\(", line):
            for j in range(i+1, min(i+3, len(lines))):
                next_line = lines[j].strip()
                match = re.match(r"^([A-Z]{2}[A-Z0-9\-]+)\s*\)", next_line)
                if match:
                    sku = match.group(1).strip()
                    sku = re.sub(r"\s+\d+$", "", sku)
                    return sku
    
    # Pattern 5: Standalone SKU with HM/LG/P prefix
    for line in full_text.splitlines():
        if any(skip in line for skip in ["Khasra", "Village", "Triplicate", "*ASSPL"]):
            continue
        match = re.search(r"\b((?:HM|LG|P)\d{3,}(?:-[A-Z0-9]+)?)\b", line)
        if match:
            sku = match.group(1)
            if re.search(r"B[A-Z0-9]{9}", line) or "₹" in line:
                return sku
    
    # Pattern 6: Gardena model numbers
    in_description = False
    for line in full_text.splitlines():
        if "Description" in line and ("Qty" in line or "Unit" in line):
            in_description = True
            continue
        if in_description:
            match = re.search(r"(?:Gardena|GARDENA)[^\n]*?(\d{5}-\d+)", line)
            if match:
                return match.group(1)
            if line.strip().startswith(("HSN", "TOTAL")):
                break
    
    return ""

In [72]:
# Cell 5: Item Row Extraction (Unit Price, Quantity, Net Amount, Total Amount)

def extract_item_row_values(full_text):
    """
    Extract Unit Price, Quantity, Net Amount, Total Amount from invoice item table.
    Handles formats with and without discount column.
    """
    lines = [l.strip() for l in full_text.splitlines() if l.strip()]
    
    for line in lines:
        lower_line = line.lower()
        if lower_line.startswith(("shipping", "total", "hsn:")):
            continue
        
        money = re.findall(r"-?₹\s*([0-9,]+\.\d{2})", line)
        
        if len(money) >= 4:
            money = [clean_amount(m) for m in money]
            
            # Find quantity: integer between ₹ values
            qty_match = re.search(r"₹[0-9,]+\.\d{2}\s+(\d+)\s+₹", line)
            quantity = int(qty_match.group(1)) if qty_match else 1
            
            unit_price = money[0]
            total_amount = money[-1]
            
            # Net Amount depends on format (with/without discount)
            if len(money) == 5:
                net_amount = money[2]  # With Discount
            elif len(money) == 4:
                net_amount = money[1]  # Without Discount
            else:
                net_amount = unit_price * quantity
            
            return unit_price, quantity, net_amount, total_amount
    
    return 0.0, 0, 0.0, 0.0

In [73]:
# Cell 6: Name Extraction Function

def extract_name_from_text(full_text):
    """
    Extract Name1 and Name2 from the shipping address section.
    Handles names on PAN No line, GST line, or separate lines.
    """
    lines = full_text.splitlines()
    
    def is_address_line(line):
        """Check if a line looks like an address (not a name)"""
        upper = line.upper()
        if re.search(r"\b\d{6}\b", line):
            return True
        address_patterns = [
            r"\bROAD\b", r"\bSTREET\b", r"\bLANE\b", r"\bFLOOR\b", r"\bBLOCK\b",
            r"\bPLOT\b", r"\bHOUSE\b", r"\bFLAT\b", r"\bNAGAR\b", r"\bCOLONY\b",
            r"\bSECTOR\b", r"\bAPARTMENT\b", r"\bBUILDING\b", r"\bCOMPLEX\b",
            r"\bNEAR\b", r"\bOPP\b", r"\bBEHIND\b", r"\bMAIN\b", r"\bCROSS\b",
            r"\bSTATION\b", r"\bVILLA\b", r"\bAPT\b", r"\bPOST\b", r"\bP\.O\b",
            r"\bNH\s", r"\bNO\.\s*\d", r"\bNO\s*\d"
        ]
        for pattern in address_patterns:
            if re.search(pattern, upper):
                return True
        if re.match(r"^[A-Z\s]+,\s*[A-Z\s]+,\s*\d{6}$", line):
            return True
        return False
    
    for i, line in enumerate(lines):
        if "Shipping Address" in line:
            for k in range(i + 1, min(i + 6, len(lines))):
                next_line = lines[k].strip()
                if not next_line:
                    continue
                
                # Check PAN No line
                if "PAN No" in next_line:
                    match = re.search(r"PAN No\s*:\s*[A-Z0-9]+\s+(.+)$", next_line)
                    if match:
                        name = match.group(1).strip()
                        if name and "Shipping" not in name:
                            parts = name.split()
                            if len(parts) >= 2:
                                return parts[0], " ".join(parts[1:])
                            elif len(parts) == 1:
                                return parts[0], ""
                    continue
                
                # Check GST Registration line
                if "GST Registration" in next_line:
                    match = re.search(r"GST Registration No\s*:\s*[A-Z0-9]+\s+(.+)$", next_line)
                    if match:
                        name = match.group(1).strip()
                        if name and len(name) > 2 and "Shipping" not in name:
                            parts = name.split()
                            if len(parts) >= 2:
                                return parts[0], " ".join(parts[1:])
                            elif len(parts) == 1:
                                return parts[0], ""
                    continue
                
                skip_keywords = ["Dynamic QR", "State/UT", "Place of", "Order Number", 
                                 "Invoice", "TOTAL", "HSN"]
                if any(kw in next_line for kw in skip_keywords):
                    continue
                if next_line.strip() == "IN":
                    continue
                if is_address_line(next_line):
                    continue
                
                parts = next_line.split()
                if 1 <= len(parts) <= 5:
                    name1 = parts[0].rstrip(",")
                    name2 = " ".join(parts[1:]).rstrip(",") if len(parts) > 1 else ""
                    return name1, name2
            break
    
    return "", ""

In [74]:
# Cell 7: Main Extraction Function

def extract_invoice_row(pdf_path):
    """Extract all fields from a single invoice PDF."""
    full_text = extract_text(pdf_path)

    # Invoice number & date
    invoice_no_match = re.search(r"Invoice Number\s*:\s*([A-Z0-9\-]+)", full_text)
    invoice_date_match = re.search(r"Invoice Date\s*:\s*([0-9\.]+)", full_text)
    invoice_no = invoice_no_match.group(1) if invoice_no_match else ""
    invoice_date = invoice_date_match.group(1) if invoice_date_match else ""

    # SKU
    sku = extract_sku(full_text)

    # Item values
    unit_price, quantity, net_amount, total_amount = extract_item_row_values(full_text)

    # Shipping address block
    ship_block = re.search(
        r"Shipping Address\s*:\s*(.*?)State/UT Code",
        full_text, re.DOTALL | re.IGNORECASE
    )
    shipping_address = ""
    if ship_block:
        lines = []
        for line in ship_block.group(1).splitlines():
            if not line.strip().startswith(("PAN No", "GST Registration", "Dynamic QR")):
                lines.append(line.strip())
        shipping_address = "\n".join(lines).strip()

    # Pincode
    pincode_match = re.search(r"\b\d{6}\b", shipping_address)
    pincode = pincode_match.group(0) if pincode_match else ""

    # City & State
    city, state = "", ""
    for line in shipping_address.splitlines():
        if pincode and pincode in line:
            parts = [p.strip() for p in line.split(",")]
            if len(parts) >= 3:
                city = parts[0]
                state = parts[1]

    # Name
    name1, name2 = extract_name_from_text(full_text)

    # Phone
    phone_match = re.search(r"\b[6-9]\d{9}\b", full_text)
    phone = phone_match.group(0) if phone_match else ""

    return {
        "Invoice No.": invoice_no,
        "File Name": pdf_path.name,
        "Invoice Date": invoice_date,
        "SKU": sku,
        "Unit Price": unit_price,
        "Quantity": quantity,
        "Net Amount": net_amount,
        "Total Amount": total_amount,
        "Shipping Address": shipping_address,
        "State": state,
        "City": city,
        "Pincode": pincode,
        "Name1": name1,
        "Name2": name2,
        "Phone": phone
    }

In [75]:
# Cell 8: Process All PDFs

rows = []
failed = []

print(f"Processing {len(pdf_files)} PDFs...")
for i, pdf_path in enumerate(pdf_files):
    try:
        row = extract_invoice_row(pdf_path)
        rows.append(row)
    except Exception as e:
        failed.append({"File Name": pdf_path.name, "Error": str(e)})
    
    # Progress indicator
    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{len(pdf_files)}")

print(f"\n✓ Successfully extracted: {len(rows)}")
print(f"✗ Failed: {len(failed)}")

Processing 1470 PDFs...


  Processed 100/1470
  Processed 200/1470
  Processed 300/1470
  Processed 400/1470
  Processed 500/1470
  Processed 600/1470
  Processed 700/1470
  Processed 800/1470
  Processed 900/1470
  Processed 1000/1470
  Processed 1100/1470
  Processed 1200/1470
  Processed 1300/1470
  Processed 1400/1470

✓ Successfully extracted: 1470
✗ Failed: 0


In [76]:
# Cell 9: Create DataFrame & Preview

df = pd.DataFrame(rows)

# Column order
columns = [
    "Invoice No.", "File Name", "Invoice Date", "SKU",
    "Unit Price", "Quantity", "Net Amount", "Total Amount",
    "Shipping Address", "State", "City", "Pincode",
    "Name1", "Name2", "Phone"
]
df = df[columns]

# Show preview
print("Preview of extracted data:")
df.head(10)

Preview of extracted data:


,Invoice No.,File Name,Invoice Date,SKU,Unit Price,Quantity,Net Amount,Total Amount,Shipping Address,State,City,Pincode,Name1,Name2,Phone
0,AMD2-48,AMD2-48_94190772556302_SHIPMENT_supplier_copy.PDF,09.06.2025,HM0513,1016.10,1,1016.10,1199.00,"G415 Bhindi Moll, Gudi Chandor road\nChandor S...",GOA,Chandor Salcette,403714,Cavan,,
1,AMD2-49,AMD2-49_94331909643302_SHIPMENT_supplier_copy.PDF,12.06.2025,LG0789,596.43,1,596.43,668.00,"Achyutham, Near police station\nELAMPALLOOR, K...",KERALA,ELAMPALLOOR,691501,Jayachandran,L,
2,AMD2-50,AMD2-50_94330167833302_SHIPMENT_supplier_copy.PDF,12.06.2025,LG0743,9995.54,1,9995.54,11195.00,"59, Anmol Enclave\nAMRITSAR, PUNJAB, 143001\nIN",PUNJAB,AMRITSAR,143001,Ashu,,
3,AMD2-51,AMD2-51_94406400316302_SHIPMENT_supplier_copy.PDF,14.06.2025,LG0743,9995.54,1,9995.54,11195.00,Manzoor ul haq\nHouse 61 gulbahar colony Hyder...,JAMMU & KASHMIR,SRINAGAR,190014,Manzoor,ul haq,
4,AMD2-52,AMD2-52_94412914922302_SHIPMENT_supplier_copy.PDF,15.06.2025,HM0532,541.98,1,525.72,620.35,"surya singh\n83,paschim vihar,durga lane ,behi...",rajasthan,jaipur,302021,WOLF,,
5,AMD2-53,AMD2-53_94424273993302_SHIPMENT_supplier_copy.PDF,15.06.2025,LG0743,9995.54,1,9995.54,11195.00,"House no 1124, Sector 71\nMohali, PUNJAB, 1600...",PUNJAB,Mohali,160071,Sanjeet,singh randhawa,
6,AMD2-54,AMD2-54_94605918726302_SHIPMENT_supplier_copy.PDF,19.06.2025,HM0531,779.42,1,779.42,919.71,"KIRTI VARDHAN SINGH\nKirti Vardhan Singh\n23, ...",DELHI,New Delhi,110001,KIRTI,VARDHAN SINGH,
7,BLR7-100,BLR7-100_94683925719302_SHIPMENT_supplier_copy...,20.06.2025,HM0526,422.88,1,422.88,499.00,"Prashant Kumar\nBENGALURU, KARNATAKA, 560060\nIN",KARNATAKA,BENGALURU,560060,Prashant,Kumar,
8,BLR7-101,BLR7-101_94695819236302_SHIPMENT_supplier_copy...,21.06.2025,HM0526,422.88,1,422.88,499.00,Flat No. 303 Bhoomi Samarth Apartment; A wing;...,MAHARASHTRA,MUMBAI,400063,Dr,Harish V.,
9,BLR7-102,BLR7-102_94709657629302_SHIPMENT_supplier_copy...,21.06.2025,HM0526,422.88,1,422.88,499.00,"A3-1207, Mahaveer Ranches, Naganathapura,\nhos...",KARNATAKA,Bengaluru,560100,Rajesh,Kumar,


In [77]:
# Cell 10: Data Quality Check

print("=== Data Quality Summary ===\n")
print(f"Total invoices: {len(df)}")
print(f"Empty SKUs: {len(df[df['SKU'] == ''])} ({len(df[df['SKU'] == ''])/len(df)*100:.1f}%)")
print(f"Empty Names: {len(df[df['Name1'] == ''])} ({len(df[df['Name1'] == ''])/len(df)*100:.1f}%)")
print(f"Zero Total Amount: {len(df[df['Total Amount'] == 0])}")

print("\n=== SKU Distribution (Top 15) ===")
print(df["SKU"].value_counts().head(15))

=== Data Quality Summary ===

Total invoices: 1470
Empty SKUs: 24 (1.6%)
Empty Names: 6 (0.4%)
Zero Total Amount: 0

=== SKU Distribution (Top 15) ===
SKU
LG0810-12    84
LG0637       62
HM0532       60
HM0526       58
LG0737       58
HM0345       46
HM0628       42
HM0484       30
LG0743       26
LG0993       26
LG0817-1     24
             24
LG0789       24
HM0492       24
HM0550       20
Name: count, dtype: int64


In [ ]:
# Cell 11: Export to Excel

output_path = Path("amazon_invoices_extracted.xlsx")
df.to_excel(output_path, index=False)

print(f"✓ Excel file saved to: {output_path}")
print(f"  Total rows: {len(df)}")

✓ Excel file saved to: Amazon Data Extraction\Amz June\bulkInvoice_2025_06_29 12_00_00 AM IST - 2025_06_30 11_59_59 PM IST\amazon_invoices_extracted.xlsx
  Total rows: 1470


In [79]:
# Cell 12: Show Failed Files (if any)

if failed:
    print(f"=== {len(failed)} Failed Files ===")
    for f in failed[:10]:
        print(f"  {f['File Name']}: {f['Error']}")
    if len(failed) > 10:
        print(f"  ... and {len(failed) - 10} more")
else:
    print("✓ All files processed successfully!")

✓ All files processed successfully!
